# Phase 2: Generative AI

## 1. Generative AI Model Selection & Setup

### Model Selection

Google Gemini 3.5 Flash was selected as the
generative AI model for our credit card default
prediction system.

The model was selected for its text generation
capabilities, ease of integration with Python,
and suitability for generating executive-level
explanations and risk summaries.

The API connection and text generation were
successfully tested before implementing the
prompt templates.

In [1]:
%pip install google-genai

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 241 kB 632 kB/s eta 0:00:01
     |████████████████████████████████| 472 kB 3.2 MB/s eta 0:00:01
     |████████████████████████████████| 246 kB 654 kB/s eta 0:00:01
     |████████████████████████████████| 73 kB 1.9 MB/s eta 0:00:01
     |████████████████████████████████| 173 kB 2.3 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 4.8 MB/s eta 0:00:011
     |████████████████████████████████| 113 kB 1.2 MB/s eta 0:00:01
     |████████████████████████████████| 69 kB 2.8 MB/s eta 0:00:01
     |████████████████████████████████| 181 kB 4.5 MB/s eta 0:00:01
     |████████████████████████████████| 3.9 MB 5.1 MB/s eta 0:00:01
     |████████████████████████████████| 180 kB 4.1 MB/s eta 0:00:01
     |████████████████████████████████| 118 kB 2.4 MB/s eta 0:00:01
     |████████████████████████████████| 127 kB 4.0 MB/s eta 0:00:01
     |████████████████████████████████| 

### API Setup and Secure Key Handling

The Google Gen AI SDK is used to integrate the
Gemini API with Python.

The API key is entered securely at runtime using
the getpass module, rather than hardcoding it
in the notebook.

This approach prevents the API key from being
stored in the source code or exposed in the
GitHub repository.

In [2]:
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)

/Users/layanalshamer/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/layanalshamer/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/layanalshamer/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Pyt

### API Connection and Model Testing

The API connection was verified by retrieving
the list of available generative AI models.

Gemini 3.5 Flash was then tested using a simple
credit card default risk question to verify
its text generation capabilities.

The model successfully generated a response,
confirming that the API integration is functional.

In [3]:
for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)
        

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemi

In [7]:
MODEL_NAME = "gemini-3.5-flash"
print("Using model:", MODEL_NAME)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain credit card default risk in two sentences."
)

print(response.text)


Using model: gemini-3.5-flash


Credit card default risk is the likelihood that a borrower will fail to make their required payments and repay their credit card balance as agreed. When a default occurs, lenders face financial losses, while the delinquent borrower suffers severe credit score damage and restricted access to future loans.


## 2. Prompt Template 1: Executive Credit Risk Briefing

### 2.1 Loading Clustering Results

In [8]:
import json
from pathlib import Path

ROOT = Path.cwd()

if ROOT.name == "Generative_AI":
    ROOT = ROOT.parent

profiles_path = (
    ROOT / "Unsupervised_Learning"
    / "clustering_results"
    / "cluster_profiles.json"
)

with open(profiles_path, "r", encoding="utf-8") as file:
    cluster_profiles = json.load(file)

print(json.dumps(cluster_profiles, indent=2))

{
  "0": {
    "segment_name": "Active On-Time Payers",
    "risk_level": "Low",
    "description": "Clients who use their card regularly, with above-average bills and above-average payment amounts, and almost no late months.",
    "recommended_action": "Retain and grow: eligible for credit-limit increases, rewards and cross-selling. Standard monitoring.",
    "key_traits": "median credit limit NT$150,000; median average bill NT$33,790; 0.2 late months out of 6 on average",
    "clients_train": 15009,
    "share_pct": 62.5,
    "default_rate_train_pct": 13.8,
    "default_rate_test_pct": 13.3,
    "risk_vs_average": 0.62,
    "mean_silhouette": 0.391
  },
  "1": {
    "segment_name": "Low-Activity Cardholders",
    "risk_level": "Medium",
    "description": "Clients who barely use their card (very small bills, many months with no activity) despite having the highest credit limits. They pay most of what they owe, but about 1 in 3 had at least one late month.",
    "recommended_action": 

### 2.2 Prompt Template Design

**Template ID:** PT-01

**Template Name:** Executive Credit Risk Briefing

**Intended Use Case:**
Generate a concise credit risk briefing for bank
managers using the supervised learning prediction
and unsupervised learning customer segment.

**Design Rationale:**
The prompt combines the Random Forest prediction
with the K-Means customer segment profile to
provide a data-driven explanation of credit risk.

A structured output format is used to improve
clarity, consistency, and interpretability.

The generated briefing supports human
decision-making rather than replacing it.

### 2.3 Prompt Template Implementation

The prompt uses dynamic placeholders to combine
supervised learning predictions with customer
segment information from unsupervised learning.

The output follows a predefined structure to
ensure clarity and consistency across test cases.

In [9]:
PROMPT_TEMPLATE_1 = """
You are a credit risk analyst assisting bank managers.

Analyze the following machine learning results.

Supervised Learning Results:
- Default Prediction: {prediction}
- Default Probability: {default_probability}%

Unsupervised Learning Results:
- Customer Segment: {segment_name}
- Segment Risk Level: {risk_level}
- Segment Description: {description}
- Key Segment Traits: {key_traits}
- Segment Default Rate: {segment_default_rate}%
- Suggested Segment Action: {recommended_action}

Generate an executive credit risk briefing using
the following structure:

1. Risk Summary
2. Key Risk Factors
3. Customer Segment Interpretation
4. Recommended Actions

Instructions:
- Use clear and professional business language.
- Base the analysis only on the provided information.
- Do not invent facts or numerical values.
- Distinguish individual prediction probability
  from the historical segment default rate.
- Do not assume that segment traits describe
  the individual customer.
- Provide recommendations for human review,
  not final lending decisions.
"""

### 2.4 Dynamic Prompt Generation

A reusable function is implemented to populate
the prompt placeholders with the Random Forest
prediction and the corresponding K-Means
customer segment profile.

In [10]:
def build_prompt_1(prediction, default_probability, cluster_id):

    profile = cluster_profiles[str(cluster_id)]

    return PROMPT_TEMPLATE_1.format(
        prediction=prediction,
        default_probability=round(default_probability * 100, 2),
        segment_name=profile["segment_name"],
        risk_level=profile["risk_level"],
        description=profile["description"],
        key_traits=profile["key_traits"],
        segment_default_rate=profile["default_rate_train_pct"],
        recommended_action=profile["recommended_action"]
    )

### 2.5 Loading Customer Data

The clustered dataset is loaded to prepare
real test cases for Prompt Template 1.

Customer records will be used to obtain
supervised learning predictions and their
corresponding customer segments.

In [11]:
import pandas as pd

data_path = ROOT / "Dataset" / "clustered_data.csv"

clustered_data = pd.read_csv(data_path)

print("Dataset shape:", clustered_data.shape)

print("\nColumn names:")
print(clustered_data.columns.tolist())

clustered_data.head()

Dataset shape: (30000, 37)

Column names:
['ID', 'EDUCATION_0', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4', 'EDUCATION_5', 'EDUCATION_6', 'LIMIT_BAL', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'AGE', 'MONTHS_LATE', 'UTILIZATION', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'default.payment.next.month', 'split', 'kmeans_cluster', 'gmm_cluster', 'gmm_membership', 'segment_name', 'risk_level']


,ID,EDUCATION_0,EDUCATION_1,EDUCATION_2,EDUCATION_3,EDUCATION_4,EDUCATION_5,EDUCATION_6,LIMIT_BAL,PAY_AMT1,...,PAY_4,PAY_5,PAY_6,default.payment.next.month,split,kmeans_cluster,gmm_cluster,gmm_membership,segment_name,risk_level
0,22789,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.340007,0.918339,...,1.911811,0.240260,0.256087,0,train,2,2,1.000000,Chronic Late Payers,High
1,29007,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.271348,-0.816245,...,-1.530826,-1.537252,-1.489963,0,train,1,1,1.000000,Low-Activity Cardholders,Medium
2,16951,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-2.609497,-1.707251,...,0.190493,0.240260,0.256087,0,train,2,2,0.999998,Chronic Late Payers,High
3,22281,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.678790,0.764288,...,0.190493,0.240260,0.256087,1,train,0,0,1.000000,Active On-Time Payers,Low
4,11347,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.043629,-2.045917,...,0.190493,0.240260,-1.489963,0,train,1,1,1.000000,Low-Activity Cardholders,Medium


### 2.6 Selecting Test Cases

Three customer records are selected from the
test dataset, representing low, medium, and
high-risk customer segments.

This selection allows the prompt to be evaluated
across different customer risk profiles.

In [12]:
test_data = clustered_data[
    clustered_data["split"] == "test"
]

test_cases = (
    test_data.groupby("kmeans_cluster")
    .sample(n=1, random_state=42)
    .sort_values("kmeans_cluster")
)

test_cases[
    ["ID", "kmeans_cluster", "segment_name", "risk_level"]
]

,ID,kmeans_cluster,segment_name,risk_level
26078,18124,0,Active On-Time Payers,Low
24394,9727,1,Low-Activity Cardholders,Medium
24029,24891,2,Chronic Late Payers,High


### 2.7 Loading the Random Forest Model

The trained Random Forest model from Phase 1
will be reused to obtain default predictions
and estimated default probabilities for the
three selected test cases.

In [13]:
model_files = list(
    (ROOT / "Supervised_Learning").rglob("*.joblib")
)

for file in model_files:
    print(file.relative_to(ROOT))

Supervised_Learning/models/rf_model.joblib
Supervised_Learning/models/svm_model.joblib


In [14]:
import joblib

rf_path = (
    ROOT / "Supervised_Learning"
    / "models"
    / "rf_model.joblib"
)

rf_model = joblib.load(rf_path)

print("Random Forest loaded successfully!")
print("Model type:", type(rf_model))
print("Classes:", getattr(rf_model, "classes_", None))
print("Expected features:",
      getattr(rf_model, "n_features_in_", None))
print("Feature names:",
      getattr(rf_model, "feature_names_in_", None))

Random Forest loaded successfully!
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Classes: [0 1]
Expected features: 29
Feature names: ['EDUCATION_0' 'EDUCATION_1' 'EDUCATION_2' 'EDUCATION_3' 'EDUCATION_4'
 'EDUCATION_5' 'EDUCATION_6' 'LIMIT_BAL' 'PAY_AMT1' 'PAY_AMT2' 'PAY_AMT3'
 'PAY_AMT4' 'PAY_AMT5' 'PAY_AMT6' 'BILL_AMT1' 'BILL_AMT2' 'BILL_AMT3'
 'BILL_AMT4' 'BILL_AMT5' 'BILL_AMT6' 'AGE' 'MONTHS_LATE' 'UTILIZATION'
 'PAY_0' 'PAY_2' 'PAY_3' 'PAY_4' 'PAY_5' 'PAY_6']


### 2.8 Preparing Model Inputs

The selected customer records are checked
against the original preprocessed dataset
and the Random Forest feature requirements
to ensure consistent model inputs.

In [15]:
preprocessed_data = pd.read_csv(
    ROOT / "Dataset" / "preprocessed_data.csv"
)

required_features = list(rf_model.feature_names_in_)

missing_features = [
    feature for feature in required_features
    if feature not in preprocessed_data.columns
]

print("Required features:", len(required_features))
print("Missing features:", missing_features)
print("Dataset shape:", preprocessed_data.shape)
print("Columns:", preprocessed_data.columns.tolist())

Required features: 29
Missing features: []
Dataset shape: (30000, 32)
Columns: ['ID', 'EDUCATION_0', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4', 'EDUCATION_5', 'EDUCATION_6', 'LIMIT_BAL', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'AGE', 'MONTHS_LATE', 'UTILIZATION', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'default.payment.next.month', 'split']


In [16]:
selected_customers = (
    preprocessed_data.set_index("ID")
    .loc[test_cases["ID"].tolist()]
)

X_cases = selected_customers[required_features]

print("Input shape:", X_cases.shape)
print("Missing values:", X_cases.isna().sum().sum())

X_cases

Input shape: (3, 29)
Missing values: 0


,EDUCATION_0,EDUCATION_1,EDUCATION_2,EDUCATION_3,EDUCATION_4,EDUCATION_5,EDUCATION_6,LIMIT_BAL,PAY_AMT1,PAY_AMT2,...,BILL_AMT6,AGE,MONTHS_LATE,UTILIZATION,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
ID,,,,,,,,,,,,,,,,,,,,,
18124,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.703432,0.281540,0.308414,...,0.586785,-0.04705,0.105940,0.373726,0.012576,0.111849,0.139780,0.190493,2.017773,0.256087
9727,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.197951,-0.261063,-0.236421,...,-0.538867,-0.59082,-0.538890,-1.025763,-0.877791,-0.722324,-0.697642,-0.670167,-0.648496,-0.616938
24891,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.397388,0.470669,-0.861905,...,0.645364,-0.59082,2.040431,1.299575,0.012576,0.111849,1.814624,1.911811,2.017773,2.002137


In [17]:
phase1_path = ROOT / "Phase1_Supervised_Learning.ipynb"

with open(phase1_path, "r", encoding="utf-8") as file:
    phase1_notebook = json.load(file)

for cell in phase1_notebook["cells"]:
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])

        if any(term in source for term in [
            "rf_model.fit",
            "RandomForestClassifier(",
            "predict_proba",
            "threshold",
            "StandardScaler",
            "preprocessed_data.csv"
        ]):
            print(source)
            print("-" * 50)

# ===== Load the preprocessed data (prepared in Phase1_Data_Exploration) =====
data = pd.read_csv(ROOT / "Dataset" / "preprocessed_data.csv")

# The file already contains the team's train/test split in the "split" column
train = data[data["split"] == "train"]
test = data[data["split"] == "test"]

# ID is only for linking rows back to the raw data; it is not a model input
drop_cols = ["ID", TARGET, "split"]

X_train = train.drop(columns=drop_cols)
y_train = train[TARGET]
X_test = test.drop(columns=drop_cols)
y_test = test[TARGET]
test_ids = test["ID"]  # kept for the misclassification analysis later

print("Training set:", X_train.shape, "| defaults:", y_train.sum())
print("Test set:    ", X_test.shape, "| defaults:", y_test.sum())

# Safety check: stop if the split is different from the team's split
assert X_train.shape == (24000, 29) and X_test.shape == (6000, 29)
assert y_train.sum() == 5309 and y_test.sum() == 1327
print("Split verified: identical to the team's split.")
------------

In [18]:
for cell in phase1_notebook["cells"]:
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])

        for line in source.splitlines():
            if any(term in line for term in [
                "threshold",
                "thr[",
                "rf_model.fit",
                "rf_model.predict",
                "StandardScaler",
                "joblib.dump"
            ]):
                print(line)

y_pred = best_rf.predict(X_test)               # predicted class (threshold 0.5)
    "threshold": 0.5,
# ===== Decision threshold analysis =====
# (the threshold is chosen without looking at the test set)
threshold_table = pd.DataFrame(rows)
display(threshold_table.round(3))
best_threshold = threshold_table.loc[threshold_table["F1"].idxmax(), "Threshold"]
print("Threshold with the best F1 on training folds:", best_threshold)
# Report the test results for the default, the best-F1 and a high-recall threshold
thresholds_to_report = sorted({0.40, float(best_threshold), 0.50})
threshold_compare = pd.DataFrame({
    for t in thresholds_to_report
threshold_compare.round(3)
joblib.dump(best_rf, model_path, compress=3)   # compressed to keep the file small
test_metrics["tuned_threshold"] = float(best_threshold)
joblib.dump(best_svm, MODELS_DIR / "svm_model.joblib")
# how many false alarms would it make? (Illustration only, not used to pick a threshold.)
      f"(probability threshold {thr[i]:.2

### 2.9 Generating Random Forest Predictions

The trained Random Forest model is used to
generate default predictions and estimated
default probabilities for the three selected
test cases, using the original classification
threshold of 0.5 from Phase 1.

In [19]:
predictions = rf_model.predict(X_cases)

probabilities = rf_model.predict_proba(X_cases)[:, 1]

prediction_results = test_cases[
    ["ID", "kmeans_cluster", "segment_name", "risk_level"]
].copy()

prediction_results["prediction"] = predictions

prediction_results["default_probability"] = probabilities

prediction_results

,ID,kmeans_cluster,segment_name,risk_level,prediction,default_probability
26078,18124,0,Active On-Time Payers,Low,1,0.547173
24394,9727,1,Low-Activity Cardholders,Medium,0,0.498034
24029,24891,2,Chronic Late Payers,High,1,0.690066


### 2.10 Testing Prompt Template 1

Prompt Template 1 is evaluated using three
real test cases representing different
customer segments and risk levels.

The Random Forest predictions, estimated
default probabilities, and cluster profiles
are provided as context to the generative
AI model to produce executive-level
explanations.

In [20]:
prompt_1_results = []

for i, (_, row) in enumerate(
    prediction_results.iterrows(), start=1
):
    prediction_label = (
        "Default" if row["prediction"] == 1
        else "No Default"
    )

    prompt = build_prompt_1(
        prediction=prediction_label,
        default_probability=float(row["default_probability"]),
        cluster_id=int(row["kmeans_cluster"])
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    prompt_1_results.append({
        "case": i,
        "customer_id": int(row["ID"]),
        "prompt": prompt,
        "output": response.text
    })

    print(f"Test Case {i}")
    print(response.text)
    print("=" * 70)

Test Case 1
# Executive Credit Risk Briefing

## 1. Risk Summary
* **Individual Prediction:** Default Predicted (Default Probability: **54.72%**)
* **Customer Segment Assignment:** Active On-Time Payers (Segment Risk Level: **Low**)
* **Historical Segment Default Rate:** **13.8%**

**Overview:** 
There is a notable conflict between the machine learning models. The supervised learning model predicts an elevated individual risk of default (**54.72%**), whereas the unsupervised model assigns the customer to the "Active On-Time Payers" segment, which carries a historically low segment default rate of **13.8%**. 

## 2. Key Risk Factors
* **Elevated Individual Default Probability:** The supervised model evaluates this specific borrower at a **54.72%** probability of default, crossing the threshold to classify the account as a predicted default.
* **Model Divergence:** The primary risk driver is the discrepancy between the customer's high individual risk score and the broad profile of their 

Test Case 2
# Executive Credit Risk Briefing

## 1. Risk Summary
* **Supervised Prediction:** No Default
* **Individual Default Probability:** 49.8%
* **Unsupervised Segment:** Low-Activity Cardholders
* **Segment Risk Level:** Medium
* **Historical Segment Default Rate:** 21.5%

**Overview:**  
While the supervised machine learning model outputs a binary classification of "No Default," the individual’s default probability stands at 49.8%. This places the account on the borderline of a default classification. Notably, there is a clear distinction between the individual’s predicted default probability (49.8%) and the broader historical segment default rate (21.5%). While the account falls into a medium-risk cohort, the individual model indicates a significantly higher risk score that warrants manual scrutiny.

---

## 2. Key Risk Factors
* **Elevated Individual Risk Score:** The 49.8% default probability indicates substantial individual risk, positioned barely below the standard thresho

Test Case 3
# Executive Credit Risk Briefing

## 1. Risk Summary
The machine learning evaluation identifies this account as high risk. The supervised learning model predicts an individual outcome of **Default** with a **69.01% default probability**. Concurrently, the unsupervised model places the customer in the **Chronic Late Payers** segment, a group designated with a **High** risk level and a historical **segment default rate of 57.6%**. 

*Note: The individual model prediction (69.01% default probability) evaluates this specific customer's risk, whereas the segment default rate (57.6%) reflects the historical baseline risk of the broader peer group.*

---

## 2. Key Risk Factors
* **Elevated Individual Default Probability:** The supervised model evaluates the individual customer's likelihood of default at 69.01%, classifying the account as a predicted default.
* **High-Risk Segment Classification:** Placement in the "Chronic Late Payers" cohort indicates alignment with a group that

### 2.11 Saving Prompt Template 1 Outputs

The generated prompts and responses for all
three test cases are saved for documentation,
evaluation, and reproducibility.

In [21]:
output_path = (
    ROOT / "Generative_AI"
    / "prompt_1_results.json"
)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(
        prompt_1_results,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Number of completed test cases:", len(prompt_1_results))
print("Results saved to:", output_path.relative_to(ROOT))

Number of completed test cases: 3
Results saved to: Generative_AI/prompt_1_results.json


### 2.12 Analysis and Evaluation of Prompt Template 1

Prompt Template 1 was successfully tested using
three real customer cases representing low,
medium, and high-risk customer segments.

#### Case 1: Low-Risk Segment

The model correctly reported a default prediction
with a default probability of 54.72%, despite the
customer belonging to a low-risk segment.

The generated response identified the difference
between individual risk and segment-level risk.
However, it suggested possible individual risk
drivers without sufficient supporting evidence.

#### Case 2: Medium-Risk Segment

The model correctly reported a No Default
prediction with a default probability of 49.80%.

The response recognized that the probability was
close to the classification threshold of 50%
and highlighted the need for further review.

#### Case 3: High-Risk Segment

The model correctly reported a default prediction
with a default probability of 69.01%.

The response identified the alignment between
the individual prediction and the high-risk
customer segment.

#### Strengths

- Accurate reporting of predictions and probabilities.
- Clear and structured executive-level explanations.
- Effective integration of supervised and
  unsupervised learning results.
- Clear distinction between individual predictions
  and segment-level characteristics.

#### Limitations

- Some explanations may include unsupported assumptions.
- Individual risk drivers cannot be confirmed without
  additional customer-level evidence.
- Generated recommendations require human validation.
- Testing only three cases limits generalizability.

#### Conclusion

Prompt Template 1 successfully generated structured
executive credit risk briefings for all three
test cases.

The results demonstrate the potential of generative
AI to explain machine learning outputs in an
understandable format. However, the generated
explanations should support human decision-making
rather than replace it.